## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [45]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

### 1 & 2. Model and Chat Model 

In [58]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [65]:
def get_chat_llm (params=None):
    
    llm = ChatGoogleGenerativeAI(
        model = "gemini-2.5-flash",
        temperature=0.3,
        api_key=Model_API
    )
    
    return llm

In [66]:
llm_model = get_chat_llm()

### 3. Chat Message 

In [67]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [68]:
print(message)

content='The **BMW M3** is an iconic choice that perfectly blends high performance with sedan practicality.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e8903-c344-7f71-af1b-e28beb0677aa-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 33, 'output_tokens': 423, 'total_tokens': 456, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 404}}


### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [69]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [70]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.3, client=<google.genai.client.Client object at 0x000001C149881690>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [71]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

#### Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [72]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [75]:
response = llm_model.invoke(formatted_prompt)
response

AIMessage(content="The world's largest mountain, by elevation above sea level, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is part of the Himalayas, located on the border between Nepal and China (Tibet Autonomous Region).\n\n---\n\nHere's a short description of the Toyota Supra:\n\nThe **Toyota Supra** is an iconic sports car produced by Toyota, renowned for its **performance, sleek styling, and powerful inline-six engines**. Historically, it's been a front-engine, rear-wheel-drive grand tourer/sports car. The **fourth-generation (A80), produced from 1993-2002**, is particularly legendary, largely due to its **2JZ-GTE twin-turbo engine** and its prominent role in popular culture (like the *Fast & Furious* franchise), making it a highly sought-after platform for tuning and aftermarket modifications. After a long hiatus, Toyota revived the Supra with the **fifth-generation (A90) in 2019**, developed in collaboration with BMW, continuing its 

In [76]:
print(response.content)

The world's largest mountain, by elevation above sea level, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is part of the Himalayas, located on the border between Nepal and China (Tibet Autonomous Region).

---

Here's a short description of the Toyota Supra:

The **Toyota Supra** is an iconic sports car produced by Toyota, renowned for its **performance, sleek styling, and powerful inline-six engines**. Historically, it's been a front-engine, rear-wheel-drive grand tourer/sports car. The **fourth-generation (A80), produced from 1993-2002**, is particularly legendary, largely due to its **2JZ-GTE twin-turbo engine** and its prominent role in popular culture (like the *Fast & Furious* franchise), making it a highly sought-after platform for tuning and aftermarket modifications. After a long hiatus, Toyota revived the Supra with the **fifth-generation (A90) in 2019**, developed in collaboration with BMW, continuing its legacy as a performance-o

In [77]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content)

The world's largest mountain is **Mount Everest**. It stands at an elevation of **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China.

---

The **Toyota Supra** is a legendary Japanese sports car and grand tourer, renowned for its performance, rear-wheel-drive dynamics, and powerful inline-six engines. It gained iconic status, particularly with the fourth-generation (Mk4) model produced from 1993-2002, which featured the highly tunable 2JZ-GTE engine and became a cultural icon, notably through its appearance in the 'Fast & Furious' film franchise. After a long hiatus, the Supra returned with its fifth generation (Mk5) in 2019, developed in collaboration with BMW, continuing its legacy as a driver-focused, high-performance machine.


### 5. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [78]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [79]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [80]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:


### 6. Output Parsers

Output parsers are used to convert the raw responses generated by an LLM into a more structured and useful format. They are especially helpful when working with structured data or when standardizing outputs from different language and chat models.

LangChain provides a variety of output parsers to handle different formatting needs. In this lab, you will work with the following two parsers:

* JSON Parser: Produces output in JSON format according to a specified schema. It can also utilize a Pydantic model to generate structured JSON data, making it one of the most dependable options for obtaining structured output without relying on function calling.

* CSV Parser: Formats the model's output as a list of comma-separated values (CSV), which is useful for tabular data representation.

In [81]:
class Car(BaseModel):
    query:str = Field(description="question to setup a car")
    response:str = Field(description="answer to resolve the car")

In [82]:
car_query = "tell me the information about Lambogini"

output_parser = JsonOutputParser(pydantic_object=Car)
format_instruction = output_parser.get_format_instructions()

prompt  = PromptTemplate(
    template= "Answer the user query.\n {format_instruction}\n{query}",
    input_variables=['query'],
    partial_variables= {"format_instruction": format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":car_query })
print(response)


{'query': 'tell me the information about Lambogini', 'response': "Lamborghini, officially Automobili Lamborghini S.p.A., is an Italian brand and manufacturer of luxury sports cars and SUVs based in Sant'Agata Bolognese. The company is owned by the Volkswagen Group through its subsidiary Audi. Founded by Ferruccio Lamborghini in 1963, it was created to compete with established marques like Ferrari. Lamborghini is renowned for its distinctive, often angular designs and high-performance engines, producing iconic models such as the Miura, Countach, Diablo, Murciélago, Gallardo, Aventador, Huracán, and the Urus SUV."}


In [83]:
user_query = "Give me details about DPO fine-tune"

output_parser = CommaSeparatedListOutputParser()
format_instruction = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query\n{format_instruction}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instruction":format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":user_query})
print(response)

['Direct Preference Optimization', 'fine-tuning LLMs', 'aligns models with human preferences', 'simpler than RLHF', 'no separate reward model needed', 'avoids PPO algorithm', 'uses human preference data', 'data consists of preferred and dispreferred response pairs', 'directly optimizes policy', 'maximizes likelihood of preferred responses', 'minimizes likelihood of dispreferred responses', 'based on Bradley-Terry model', 'uses a simple loss function', 'computationally efficient', 'more stable training', 'often achieves comparable or better performance than RLHF', 'requires a reference model (e.g.', 'SFT)']


## **LangChain Components for RAG Application**

### 7. Documents

#### Document Object

The document object contains two main components:

1. page-content: this represents the content of the whole document.
2. metadata: this represents the all the attributes that paticularlly unique for the each document. this availables at Dict types

examples:
document_id, timestamp, document_title, author, etc

In [84]:
document = Document(
    page_content="This is a example for document attribute which is page_content",
    metadata={
        "document_id": 1294857,
        "document_source": "about firewall",
        "created_time": 21344656
    }
)

document.page_content

'This is a example for document attribute which is page_content'

#### Document Loaders

Document loaders in LangChain are used to import data from multiple sources. For example, you can load a PDF file and make it readable for an LLM using LangChain.

LangChain provides more than 100 different document loaders, along with integrations with major platforms like AirByte and Unstructured. These integrations allow you to load various types of content such as HTML, PDFs, and source code from different locations, including private S3 buckets and public websites.

A full list of supported document types can be found here: https://python.langchain.com/v0.1/docs/integrations/document_loaders/

In [85]:
file_path = "https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf"
loader = PyPDFLoader(file_path)

In [86]:
loader

In [87]:
document = loader.load()
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tool

#### URL and Website Loaders

In [88]:
web_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()

In [89]:
web_data[0].page_content[:1000]

'LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.Copy pageDocumentation IndexFetch the comp

#### Text-Splitter

After loading documents, you often need to transform them so they work better for your specific use case.

A common step is breaking a long document into smaller chunks that can fit within a model’s context window. LangChain provides several built-in document transformers that help with splitting, merging, filtering, and other ways of processing documents.

In general, text splitters work in the following way:

1. The text is first divided into small, meaningful units, such as sentences.
2. These small parts are then gradually combined into larger chunks until a defined size limit is reached (based on a chosen measurement method).
3. Once a chunk reaches the limit, it is stored as a separate segment, and a new chunk begins. Often, a small overlap is kept between chunks to preserve context across sections.

You can view the different types of text splitters supported by LangChain here:
[https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/)

As an example, we will use the simple `CharacterTextSplitter` to divide the LangChain paper you previously loaded. This basic splitter works by breaking text using characters (defaulting to `"\n\n"`) and measuring chunk size based on the number of characters.


In [103]:
text_splitter = CharacterTextSplitter(chunk_size = 700, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))

60


In [104]:
print(chunks[8].page_content)

deployment of LangChain applications. Finally, section 5 addresses the limita-
tions and criticisms of LangChain, particularly focusing on the complexities and
security concerns associated with its modular design and external integrations.
1 Architecture
LangChain is built with a modular architecture, designed to simplify the life-
cycle of applications powered by large language models (LLMs), from initial
development through to deployment and monitoring [3]. This modularity al-
lows developers to configure, extend, and deploy applications tailored to specific


#### Embedding Models

Embedding models are designed to work directly with text embeddings.

They convert a piece of text into a numerical vector representation. This is useful because it lets you represent text in a vector space, where similar meanings are positioned closer together. As a result, you can perform tasks like semantic search by finding text segments that are closest to each other in that vector space.


In [105]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    api_key=Model_API
)

In [106]:
text = "Hi im seran"
embeds = embeddings.embed_query("Hi im seran")
embeds

[0.0056103403,
 0.036686193,
 0.007968124,
 -0.012751826,
 0.020288246,
 -0.011948124,
 0.009202696,
 0.013190522,
 0.0022924612,
 -0.036529027,
 -0.002601864,
 -0.004427653,
 -0.0040804255,
 0.00012383792,
 0.00059155736,
 -0.024476398,
 0.03420043,
 -0.0002107415,
 -0.006732871,
 -0.01092944,
 0.009508564,
 -0.02845144,
 0.007161191,
 -0.0060097887,
 -0.0053270287,
 -0.012943567,
 -0.011133201,
 -0.0049616075,
 -0.03637322,
 0.12255258,
 -0.007223886,
 0.009102838,
 0.0021087567,
 -0.02021646,
 -0.0018475761,
 -0.016638352,
 0.0013085961,
 0.0022267252,
 -0.016855603,
 -0.0024761264,
 0.0059231357,
 0.0027393845,
 0.013814153,
 0.012377565,
 -0.008859692,
 0.0071116243,
 -0.014386142,
 0.0025682084,
 -0.0029223713,
 0.007140271,
 -0.0028166755,
 -0.017668193,
 0.015596635,
 -0.014943381,
 -0.0023113927,
 -0.034170672,
 0.0021437968,
 0.002174044,
 -0.030944992,
 0.0023542081,
 -0.016099293,
 0.02163537,
 0.00972708,
 -0.0062310835,
 -0.019994,
 -0.010132473,
 -0.00037567478,
 0.03415

In [108]:
## to embedding documents
text = [doc.page_content for doc in chunks]
document_embeds = embeddings.embed_documents(text[0:100])

In [109]:
print(len(document_embeds))

60


#### Vector Stores

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) takes care of storing embedded data and performing vector search for you.


We use **Chroma** here to demonstrase the vector stores

In [110]:
# 1st initiate chroma object using document chunks and embeddings
doc_search = Chroma.from_documents(chunks, embeddings)

query = "Langchain"

#if we want to find the relevant context for query
docs = doc_search.similarity_search(query)
docs

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2\nPlease retry in 37.319233744s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'q